In [1]:
import sys
from pathlib import Path

from nlp_tarifas_ica.lectura_pdf import leer_pdf
from nlp_tarifas_ica.segmentacion import segmentar_articulos
from nlp_tarifas_ica.extraccion_tablas import extraer_tabla_guayabetal
from nlp_tarifas_ica.exportacion import exportar_csv, exportar_json

print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)

BASE = Path.cwd()
if not (BASE / "pyproject.toml").is_file():
    BASE = BASE.parent

if not (BASE / "pyproject.toml").is_file():
    raise FileNotFoundError("No se encontró la raíz del proyecto.")

print("Proyecto:", BASE)

Python: 3.13.8
Intérprete: /workspaces/NLP_Tarifas_ICA/.venv/bin/python
Proyecto: /workspaces/NLP_Tarifas_ICA


In [2]:
RUTA_PDF = (
    BASE / "data" / "raw"
    / "Guayabetal_2021_31Diciembre_resolucion-no-010.pdf"
)

paginas = leer_pdf(RUTA_PDF)
sin_texto = [p["pagina"] for p in paginas if not p["texto"].strip()]

print("Archivo:", RUTA_PDF.name)
print("Páginas:", len(paginas))
print("Páginas sin texto extraíble:", sin_texto)

if not any(p["texto"].strip() for p in paginas):
    raise ValueError("El PDF no tiene texto extraíble. El flujo no utiliza OCR.")

Archivo: Guayabetal_2021_31Diciembre_resolucion-no-010.pdf
Páginas: 16
Páginas sin texto extraíble: []


In [3]:
# Extractor específico para el formato de Guayabetal.
registros = extraer_tabla_guayabetal(RUTA_PDF)

print("Candidatos extraídos:", len(registros))
print(
    "Candidatos sin tarifa:",
    sum(r["tarifa_candidata"] is None for r in registros),
)

import pandas as pd

df_candidatos = pd.DataFrame(registros)
df_candidatos.head()

Candidatos extraídos: 439
Candidatos sin tarifa: 18


,archivo,pagina,codigo_actividad,ciiu,descripcion_parcial,tarifa_candidata,unidad_candidata,evidencia,estado,requiere_revision
0,Guayabetal_2021_31Diciembre_resolucion-no-010.pdf,1,101,1011,Procesamiento y conservación de carne y productos,5,por_mil,101 1011 Procesamiento y conservación de carne...,pendiente_revision,True
1,Guayabetal_2021_31Diciembre_resolucion-no-010.pdf,2,101,1012,"Procesamiento y Conservación de pescados,",5,por_mil,101 1012 Procesamiento y Conservación de pesca...,pendiente_revision,True
2,Guayabetal_2021_31Diciembre_resolucion-no-010.pdf,2,101,1020,"Procesamiento y conservación de frutas, legumb...",5,por_mil,101 1020 Procesamiento y conservación de fruta...,pendiente_revision,True
3,Guayabetal_2021_31Diciembre_resolucion-no-010.pdf,2,101,1030,,5,por_mil,101 1030 5,pendiente_revision,True
4,Guayabetal_2021_31Diciembre_resolucion-no-010.pdf,2,101,1040,Elaboración de .roductos lácteos.,5,por_mil,101 1040 Elaboración de .roductos lácteos. 5,pendiente_revision,True


In [4]:
SALIDA = BASE / "data" / "processed"

ruta_json = exportar_json(
    registros, SALIDA / "candidatos_guayabetal.json"
)

if registros:
    ruta_csv = exportar_csv(
        registros, SALIDA / "candidatos_guayabetal.csv"
    )
    print("CSV:", ruta_csv)

print("JSON:", ruta_json)
print("Registros exportados:", len(registros))
print("Estado: candidatos pendientes de revisión humana.")

CSV: /workspaces/NLP_Tarifas_ICA/data/processed/candidatos_guayabetal.csv
JSON: /workspaces/NLP_Tarifas_ICA/data/processed/candidatos_guayabetal.json
Registros exportados: 439
Estado: candidatos pendientes de revisión humana.


In [5]:
import csv
from nlp_tarifas_ica.validacion import evaluar_extraccion

RUTA_REFERENCIA = BASE / "data" / "referencia" / "referencia_manual.csv"

with RUTA_REFERENCIA.open(
    encoding="utf-8-sig", newline=""
) as archivo:
    referencia = list(csv.DictReader(archivo))

metricas = evaluar_extraccion(
    referencia=referencia,
    candidatos=registros,
    paginas_evaluadas=[(RUTA_PDF.name, 9)],
)

print("Evaluación de desarrollo: Guayabetal, página 9")
print("Verdaderos positivos:", metricas["verdaderos_positivos"])
print("Falsos positivos:", metricas["falsos_positivos"])
print("Falsos negativos:", metricas["falsos_negativos"])
print("Candidatos sin tarifa:", metricas["candidatos_sin_tarifa"])

for nombre in ("precision", "recall", "f1"):
    valor = metricas[nombre]
    resultado = f"{valor:.2%}" if valor is not None else "No definida"
    print(f"{nombre}: {resultado}")

print("\nNo se evalúan las descripciones.")
print("Estos resultados corresponden únicamente a esta página.")

Evaluación de desarrollo: Guayabetal, página 9
Verdaderos positivos: 31
Falsos positivos: 0
Falsos negativos: 4
Candidatos sin tarifa: 4
precision: 100.00%
recall: 88.57%
f1: 93.94%

No se evalúan las descripciones.
Estos resultados corresponden únicamente a esta página.


## Alcance y limitaciones

- Se procesa texto extraíble de PDF, sin OCR.
- El extractor de tablas está configurado para el formato de Guayabetal.
- Los resultados son candidatos pendientes de revisión humana.
- Algunas descripciones están incompletas o vacías y algunas tarifas no se recuperan.
- El número de candidatos no mide precisión ni cobertura.
- La evaluación con precisión, recall y F1 requiere una referencia manual independiente.